# EEG-to-Speech Architecture Tutorial

## 0. Setup & Imports

In [2]:
import os
import sys
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

%matplotlib inline

# ── Project paths ──
PROJECT_ROOT = os.path.abspath('.')  # Run this notebook from the project root
sys.path.insert(0, PROJECT_ROOT)

EEG_FILE = "/media/hdd1/amkapoor/N400/N400Stimset_manuscriptdata/N400Stimset_manuscriptdata/N400_epoched/sub-01/sub-01-_-NPC_aisle.wav.npy"
CHECKPOINT_DIR = "/media/hdd3/amkapoor/logs/subject_disc_3"
CONFIG_PATH = os.path.join(CHECKPOINT_DIR, "configs.json")

# Fallback to project config if not found at checkpoint dir
if not os.path.exists(CONFIG_PATH):
    CONFIG_PATH = os.path.join(PROJECT_ROOT, "configs", "configs.json")
    print(f"[INFO] Config not found in checkpoint dir, falling back to: {CONFIG_PATH}")

EEG_SAMPLE_RATE = 256  # Hz
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

PyTorch      : 2.9.1+cu128
CUDA         : True


## Load Raw EEG Data


In [5]:
eeg_data = np.load(EEG_FILE)

print(f"EEG shape   : {eeg_data.shape}")
print(f"Data type     : {eeg_data.dtype}")

if eeg_data.ndim == 2:
    n_channels, n_frames = eeg_data.shape
elif eeg_data.ndim == 3:
    print(f"⚠️  3D array detected (shape {eeg_data.shape}). Using first slice, last 2 dims.")
    eeg_data = eeg_data[0] if eeg_data.shape[0] > 1 else eeg_data.squeeze(0)
    n_channels, n_frames = eeg_data.shape
else:
    raise ValueError(f"Unexpected EEG shape: {eeg_data.shape}")

duration_sec = n_frames / EEG_SAMPLE_RATE

print(f"\n── EEG Signal Properties ──")
print(f"🧠 Channels      : {n_channels}")
print(f"🔢 Frames (T)    : {n_frames}")
print(f"⏱️  Sample Rate   : {EEG_SAMPLE_RATE} Hz")
print(f"⏳ Duration       : {duration_sec:.3f} seconds")

EEG shape   : (136, 668)
Data type     : float64

── EEG Signal Properties ──
🧠 Channels      : 136
🔢 Frames (T)    : 668
⏱️  Sample Rate   : 256 Hz
⏳ Duration       : 2.609 seconds


---
## 3. Load Model Architecture & Checkpoint

In [7]:
import utils
from EEGModule import EEGModule
from models import SpeechDecoder

hps = utils.get_hparams_from_file(CONFIG_PATH)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [8]:
# Build models
eeg_module = EEGModule(
    n_layers_cnn=hps.model.eeg_module.n_layers_cnn,
    use_s4=hps.model.eeg_module.use_s4,
    n_layers_s4=hps.model.eeg_module.n_layers_s4,
    embedding_size=hps.model.inter_channels,
    is_mask=False,
    in_channels=hps.model.eeg_module.in_channels,
    num_subjects=hps.model.num_subjects if hasattr(hps.model, 'num_subjects') else 25,
    device=device
).to(device)

net_g = SpeechDecoder(
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model
).to(device)

print("✅ Models built successfully")
print(f"   EEGModule params  : {sum(p.numel() for p in eeg_module.parameters()):,}")
print(f"   SpeechDecoder params: {sum(p.numel() for p in net_g.parameters()):,}")

INFO:s4_block.s4:Constructing S4 (H, N, L) = (192, 32, None)
INFO:s4_block.s4:Constructing S4 (H, N, L) = (192, 32, None)
INFO:s4_block.s4:Constructing S4 (H, N, L) = (192, 32, None)
INFO:s4_block.s4:Constructing S4 (H, N, L) = (192, 32, None)


/home/amkapoor/miniconda3/envs/speech_fixed/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


✅ Models built successfully
   EEGModule params  : 1,380,889
   SpeechDecoder params: 29,894,836


/home/amkapoor/miniconda3/envs/speech_fixed/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


In [9]:
# Find and load the latest checkpoint
ckpt_files = [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith("E_") and f.endswith(".pth")]
iterations = []
for f in ckpt_files:
    try:
        idx = int(f.replace("E_", "").replace(".pth", ""))
        iterations.append(idx)
    except ValueError:
        pass
latest_iter = max(iterations)

e_path = os.path.join(CHECKPOINT_DIR, f"E_{latest_iter}.pth")
g_path = os.path.join(CHECKPOINT_DIR, f"G_{latest_iter}.pth")

print(f"📦 Loading EEG checkpoint   : {e_path}")
print(f"📦 Loading Speech checkpoint : {g_path}")

utils.load_checkpoint(e_path, eeg_module, None)
utils.load_checkpoint(g_path, net_g, None)

eeg_module.eval()
net_g.eval()

print(f"Checkpoints loaded (iteration {latest_iter})")

📦 Loading EEG checkpoint   : /media/hdd3/amkapoor/logs/subject_disc_3/E_224000.pth
📦 Loading Speech checkpoint : /media/hdd3/amkapoor/logs/subject_disc_3/G_224000.pth
INFO:root:subject_discriminator.net.0.weight is not in the checkpoint
INFO:root:subject_discriminator.net.0.bias is not in the checkpoint
INFO:root:subject_discriminator.net.3.weight is not in the checkpoint
INFO:root:subject_discriminator.net.3.bias is not in the checkpoint
INFO:root:subject_discriminator.net.6.weight is not in the checkpoint
INFO:root:subject_discriminator.net.6.bias is not in the checkpoint
INFO:root:Loaded checkpoint '/media/hdd3/amkapoor/logs/subject_disc_3/E_224000.pth' (iteration 996)
INFO:root:enc_proj.pre_ln.weight is not in the checkpoint
INFO:root:enc_proj.pre_ln.bias is not in the checkpoint
INFO:root:enc_proj.lstm.weight_ih_l0 is not in the checkpoint
INFO:root:enc_proj.lstm.weight_hh_l0 is not in the checkpoint
INFO:root:enc_proj.lstm.bias_ih_l0 is not in the checkpoint
INFO:root:enc_proj.ls

## Forward Pass — Tracing Dimensions Through EEG Module

In [12]:
# Prepare input tensor: [1, C, T]
eeg_tensor = torch.from_numpy(eeg_data).float().unsqueeze(0).to(device)
eeg_tensor = eeg_tensor[:, :128, :].clone()
T_eeg = eeg_tensor.shape[2]
eeg_lengths = torch.tensor([T_eeg], dtype=torch.long).to(device)

print("INPUT")
print(f"EEG Tensor: {list(eeg_tensor.shape)}")
print(f"[batch=1, channels={eeg_tensor.shape[1]}, frames={T_eeg}]")
print(f"EEG Sample Rate: {EEG_SAMPLE_RATE} Hz")
print(f"EEG Duration: {T_eeg / EEG_SAMPLE_RATE:.3f} s")


INPUT
EEG Tensor: [1, 128, 668]
[batch=1, channels=128, frames=668]
EEG Sample Rate: 256 Hz
EEG Duration: 2.609 s


In [14]:

with torch.no_grad():
    x =  eeg_tensor

    # 1) Conv Encoder (6 layers, stride=1)
    conv_out = eeg_module.conv_encoder(x)
    print(f"conv_encoder output  : {list(conv_out.shape)}")
    print(f"[B, {conv_out.shape[1]}, {conv_out.shape[2]}]")


    # 2) Conv Encoder 2 (1 layer, stride=3) — temporal downsampling
    conv_out2 = eeg_module.conv_encoder2(conv_out)
    print(f"conv_encoder2 output : {list(conv_out2.shape)}")
    print(f"[B, {conv_out2.shape[1]}, {conv_out2.shape[2]}])")


    # 3) S4 Model — sequence modeling
    if eeg_module.use_s4:
        s4_input = conv_out2.transpose(-1, -2)  # [B, T, C]
        print(f"S4 input (transposed): {list(s4_input.shape)}  [B, T, d_model]")
        s4_output = eeg_module.s4_model(s4_input)
        mid_output = s4_output.transpose(1, 2)  # [B, C, T]
        print(f"S4 output            : {list(s4_output.shape)}  [B, T, d_model]")
        print(f"mid_output (trans.)   : {list(mid_output.shape)}  [B, C, T]")
    else:
        mid_output = conv_out2
        print(f"mid_output:{list(mid_output.shape)}")

    # 4) Subject Discriminator — adversarial invariance
    sid_input = mid_output.mean(dim=2)  # [B, C]
    sid_logits = eeg_module.subject_discriminator(sid_input, 1.0)
    print(f"subject_disc input   : {list(sid_input.shape)}  (mean over time)")
    print(f"subject_disc output  : {list(sid_logits.shape)}  (logits for subjects)")

    # 5) Decoder (reconstruction branch)
    dec_out = eeg_module.deconv_encoder2(mid_output)
    print(f"deconv_encoder2      : {list(dec_out.shape)}  (temporal up ×3)")
    dec_out2 = eeg_module.deconv_encoder(dec_out)
    print(f"deconv_encoder       : {list(dec_out2.shape)}  (reconstruct EEG)")

conv_encoder output  : [1, 192, 674]
[B, 192, 674]
conv_encoder2 output : [1, 192, 225]
[B, 192, 225])
S4 input (transposed): [1, 225, 192]  [B, T, d_model]
S4 output            : [1, 225, 192]  [B, T, d_model]
mid_output (trans.)   : [1, 192, 225]  [B, C, T]
subject_disc input   : [1, 192]  (mean over time)
subject_disc output  : [1, 25]  (logits for subjects)
deconv_encoder2      : [1, 192, 674]  (temporal up ×3)
deconv_encoder       : [1, 128, 668]  (reconstruct EEG)


## Forward Pass — Tracing Dimensions Through Speech Decoder

In [15]:
print("SPEECH DECODER")

with torch.no_grad():
    T_mid = mid_output.shape[2]
    mid_lengths = (eeg_lengths.float() * T_mid / T_eeg).long()
    print(f"mid_output: {list(mid_output.shape)}")
    print(f"mid_output_lengths: {mid_lengths.tolist()}")

    # ── 1) Connector (Transformer Encoder + projection) ──
    x_conn, m_p, logs_p, x_mask = net_g.enc_proj(mid_output, mid_lengths)
    print(f"Connector (Transformer Encoder)")
    print(f"encoder output: {list(x_conn.shape)}")
    print(f"m (mean): {list(m_p.shape)}")
    print(f"logs (log-variance): {list(logs_p.shape)}")
    print(f"x_mask: {list(x_mask.shape)}")

    # ── 2) Latent Sampling ──
    import commons
    y_lengths = mid_lengths.clone()
    y_mask = torch.unsqueeze(commons.sequence_mask(y_lengths, None), 1).to(x_mask.dtype)
    m_p_c = m_p[:, :, :y_mask.size(2)]
    logs_p_c = logs_p[:, :, :y_mask.size(2)]
    noise_scale = 0.667
    z_p = m_p_c + torch.randn_like(m_p_c) * torch.exp(logs_p_c) * noise_scale
    print(f"Latent Sampling")
    print(f"z_p (sampled latent) : {list(z_p.shape)}")


    z = net_g.flow(z_p, y_mask, g=None, reverse=True)
    print(f"Normalizing Flow (reverse)")
    print(f"z (decoded latent)   : {list(z.shape)}")

    max_len = 1000
    gen_input = (z * y_mask)[:, :, :max_len]
    print(f"HiFi-GAN Generator")
    print(f"generator input: {list(gen_input.shape)}")

    g_x = net_g.dec.conv_pre(gen_input)
    print(f"conv_pre: {list(g_x.shape)}")

    for i in range(net_g.dec.num_upsamples):
        g_x = torch.nn.functional.leaky_relu(g_x, 0.1)
        g_x = net_g.dec.ups[i](g_x)
        xs = None
        for j in range(net_g.dec.num_kernels):
            if xs is None:
                xs = net_g.dec.resblocks[i * net_g.dec.num_kernels + j](g_x)
            else:
                xs += net_g.dec.resblocks[i * net_g.dec.num_kernels + j](g_x)
        g_x = xs / net_g.dec.num_kernels
        rate = hps.model.upsample_rates[i]
        print(f"upsample[{i}] (×{rate:>2d})      : {list(g_x.shape)}")

    g_x = torch.nn.functional.leaky_relu(g_x)
    g_x = net_g.dec.conv_post(g_x)
    g_x = torch.tanh(g_x)
    print(f"conv_post + tanh: {list(g_x.shape)}")


SPEECH DECODER
mid_output: [1, 192, 225]
mid_output_lengths: [225]
Connector (Transformer Encoder)
encoder output: [1, 192, 225]
m (mean): [1, 192, 225]
logs (log-variance): [1, 192, 225]
x_mask: [1, 1, 225]
Latent Sampling
z_p (sampled latent) : [1, 192, 225]
Normalizing Flow (reverse)
z (decoded latent)   : [1, 192, 225]
HiFi-GAN Generator
generator input: [1, 192, 225]
conv_pre: [1, 512, 225]
upsample[0] (× 8)      : [1, 256, 1800]
upsample[1] (× 8)      : [1, 128, 14400]
upsample[2] (× 2)      : [1, 64, 28800]
upsample[3] (× 2)      : [1, 32, 57600]
conv_post + tanh: [1, 1, 57600]


# Full Inference & Output Audio Summary


In [17]:
with torch.no_grad():
    y_hat, _, mask, *_ = net_g.infer(mid_output, mid_lengths, max_len=1000, noise_scale=0.667)

audio_sr = hps.data.sampling_rate
T_audio = y_hat.shape[2]
audio_duration = T_audio / audio_sr

print(" OUTPUT AUDIO WAVEFORM")
print(f"Waveform shape: {list(y_hat.shape)}")
print(f"Audio Sample Rate: {audio_sr} Hz")
print(f"Audio Frames: {T_audio}")
print(f"Audio Duration: {audio_duration:.3f} seconds")

 OUTPUT AUDIO WAVEFORM
Waveform shape: [1, 1, 57600]
Audio Sample Rate: 22050 Hz
Audio Frames: 57600
Audio Duration: 2.612 seconds


---
## 7. Dimension Summary Table

In [18]:
print("\n┌─ COMPLETE DIMENSION SUMMARY ───────────────────────────────────────────────┐")
print(f"│  {'Stage':<30s} {'Shape':<25s} {'Note':<25s} │")
print(f"│  {'─'*30} {'─'*25} {'─'*25} │")

rows = [
    ('EEG Input',              f'[1, {eeg_tensor.shape[1]}, {T_eeg}]',      f'@ {EEG_SAMPLE_RATE} Hz'),
    ('conv_encoder (6×)',      str(list(conv_out.shape)),                    'stride=1, k=4'),
    ('conv_encoder2 (1×)',     str(list(conv_out2.shape)),                   'stride=3, k=4'),
    ('S4 mid_output',          str(list(mid_output.shape)),                  '4 S4 blocks'),
    ('Connector',              str(list(x_conn.shape)),                      '6-layer Transformer'),
    ('z_p (sampled latent)',    str(list(z_p.shape)),                         '+ noise'),
    ('z (flow reverse)',        str(list(z.shape)),                           '4× ResidualCoupling'),
    ('Generator output',       str(list(y_hat.shape)),                       f'@ {audio_sr} Hz'),
]

for stage, shape, note in rows:
    print(f"│  {stage:<30s} {shape:<25s} {note:<25s} │")

print(f"└────────────────────────────────────────────────────────────────────────────┘")

print(f"\n🧠 EEG: {n_channels} channels × {n_frames} frames @ {EEG_SAMPLE_RATE} Hz = {duration_sec:.3f}s")
print(f"🔊 Audio: 1 channel × {T_audio} frames @ {audio_sr} Hz = {audio_duration:.3f}s")


┌─ COMPLETE DIMENSION SUMMARY ───────────────────────────────────────────────┐
│  Stage                          Shape                     Note                      │
│  ────────────────────────────── ───────────────────────── ───────────────────────── │
│  EEG Input                      [1, 136, 668]             @ 256 Hz                  │
│  conv_encoder (6×)              [1, 192, 674]             stride=1, k=4             │
│  conv_encoder2 (1×)             [1, 192, 225]             stride=3, k=4             │
│  S4 mid_output                  [1, 192, 225]             4 S4 blocks               │
│  Connector                      [1, 192, 225]             6-layer Transformer       │
│  z_p (sampled latent)           [1, 192, 225]             + noise                   │
│  z (flow reverse)               [1, 192, 225]             4× ResidualCoupling       │
│  Generator output               [1, 1, 57600]             @ 22050 Hz                │
└───────────────────────────────────────

In [22]:
import IPython.display as ipd
print(f"Playing synthesized audio ({audio_duration:.2f}s) at {audio_sr} Hz...")
ipd.Audio(y_hat[0].cpu().numpy(), rate=audio_sr)

Playing synthesized audio (2.61s) at 22050 Hz...
